In [5]:
# =========================================
# 🚀 GROQ LLM-BASED CLASSIFICATION (Smart Features + LLM)
# =========================================
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
import skimage.feature as skf
import skimage.stats as sts
from groq import Groq
from dotenv import load_dotenv
from matplotlib.gridspec import GridSpec

# Load API key
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("❌ GROQ_API_KEY not found in .env file")

groq_client = Groq(api_key=api_key)

CLASS_NAMES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']

BASE_PATH = r"C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\NEU-DET"
IMAGES_DIR = os.path.join(BASE_PATH, "IMAGES")
ANNOTATIONS_DIR = os.path.join(BASE_PATH, "ANNOTATIONS")
OUTPUT_DIR = os.path.join(BASE_PATH, "OUTPUT")

# Define custom layers for model loading
class AMFF(layers.Layer):
    def __init__(self, filters, **kwargs):
        super(AMFF, self).__init__(**kwargs)
        self.filters = filters
        
    def build(self, input_shape):
        self.global_pool = layers.GlobalAveragePooling2D(keepdims=True)
        self.fc1 = layers.Dense(self.filters // 2, activation='relu')
        self.fc2 = layers.Dense(self.filters, activation='sigmoid')
        self.conv_spatial = layers.Conv2D(1, 7, padding='same', activation='sigmoid')
        self.conv_fusion = layers.Conv2D(self.filters, 1, padding='same')
        self.bn = layers.BatchNormalization()
        
    def call(self, inputs):
        high_feat, low_feat = inputs
        high_feat_resized = tf.image.resize(high_feat, tf.shape(low_feat)[1:3])
        channel_attention = self.global_pool(high_feat_resized)
        channel_attention = self.fc1(channel_attention)
        channel_attention = self.fc2(channel_attention)
        high_feat_channel = high_feat_resized * channel_attention
        spatial_map = tf.concat([
            tf.reduce_max(high_feat_resized, axis=-1, keepdims=True),
            tf.reduce_mean(high_feat_resized, axis=-1, keepdims=True)
        ], axis=-1)
        spatial_attention = self.conv_spatial(spatial_map)
        high_feat_spatial = high_feat_resized * spatial_attention
        fused = tf.concat([high_feat_channel, high_feat_spatial, low_feat], axis=-1)
        output = self.conv_fusion(fused)
        output = self.bn(output)
        output = tf.nn.relu(output)
        return output
    
    def get_config(self):
        config = super(AMFF, self).get_config()
        config.update({'filters': self.filters})
        return config

class FPN(layers.Layer):
    def __init__(self, **kwargs):
        super(FPN, self).__init__(**kwargs)
        
    def build(self, input_shape):
        F = 512
        self.lateral_c5 = layers.Conv2D(F, 1)
        self.lateral_c4 = layers.Conv2D(F, 1)
        self.lateral_c3 = layers.Conv2D(F, 1)
        self.lateral_c2 = layers.Conv2D(F, 1)
        self.amff_p4 = AMFF(F)
        self.amff_p3 = AMFF(F)
        self.amff_p2 = AMFF(F)
        self.smooth_p4 = layers.Conv2D(F, 3, padding='same')
        self.smooth_p3 = layers.Conv2D(F, 3, padding='same')
        self.smooth_p2 = layers.Conv2D(F, 3, padding='same')

    def call(self, features):
        c2, c3, c4, c5 = features['C2'], features['C3'], features['C4'], features['C5']
        p5 = self.lateral_c5(c5)
        p4 = self.amff_p4([p5, self.lateral_c4(c4)])
        p4 = self.smooth_p4(p4)
        p3 = self.amff_p3([p4, self.lateral_c3(c3)])
        p3 = self.smooth_p3(p3)
        p2 = self.amff_p2([p3, self.lateral_c2(c2)])
        p2 = self.smooth_p2(p2)
        return {'P2': p2, 'P3': p3, 'P4': p4, 'P5': p5}
    
    def get_config(self):
        return super(FPN, self).get_config()

def get_all_images(images_dir):
    """Collect all image files from IMAGES directory (including subdirectories)"""
    image_files = []
    subdirs = [d for d in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, d))]
    
    if subdirs:
        for subdir in subdirs:
            subdir_path = os.path.join(images_dir, subdir)
            for filename in os.listdir(subdir_path):
                if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
                    image_files.append({
                        'filename': filename,
                        'full_path': os.path.join(subdir_path, filename),
                        'defect_type': subdir
                    })
    else:
        for filename in os.listdir(images_dir):
            if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
                image_files.append({
                    'filename': filename,
                    'full_path': os.path.join(images_dir, filename),
                    'defect_type': None
                })
    
    return sorted(image_files, key=lambda x: x['filename'])

# Load the trained model
MODEL_PATH = r"C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\Model_Outputs_1\mobilenetv2_fpn_amff_model.h5"

print("=" * 70)
print("🔄 LOADING TRAINED MODEL (for feature extraction)")
print("=" * 70)

custom_objects = {'AMFF': AMFF, 'FPN': FPN}

try:
    trained_model = tf.keras.models.load_model(MODEL_PATH, custom_objects=custom_objects)
    print(f"✅ Model loaded successfully\n")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# =========================================
# 🧠 FEATURE EXTRACTION FROM IMAGE
# =========================================

def extract_image_features(img_rgb, img_size=128):
    """Extract texture and structural features from image"""
    
    img_resized = cv2.resize(img_rgb, (img_size, img_size))
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
    img_normalized = img_resized / 255.0
    
    features = {}
    
    # 1. TEXTURE FEATURES
    features['contrast'] = round(float(np.std(img_gray) / np.mean(img_gray + 1e-5)), 3)
    
    # Entropy
    p = np.histogram(img_gray, bins=256)[0] / img_gray.size
    p = p[p > 0]
    features['entropy'] = round(float(-np.sum(p * np.log2(p))), 3)
    
    # Edge density
    edges = cv2.Canny(img_gray, 50, 150)
    features['edge_density'] = round(float(np.sum(edges > 0) / edges.size * 100), 2)
    
    # 2. DEFECT REGION DETECTION
    _, binary = cv2.threshold(img_gray, 0, 255, cv2.THRESH_OTSU)
    binary_inv = cv2.bitwise_not(binary)
    contours, _ = cv2.findContours(binary_inv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    defect_areas = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > 100:
            defect_areas.append(area)
    
    features['num_defect_regions'] = len(defect_areas)
    features['avg_defect_area'] = round(float(np.mean(defect_areas)) if defect_areas else 0, 2)
    features['total_defect_ratio'] = round(float(np.sum(defect_areas) / (img_gray.shape[0] * img_gray.shape[1]) * 100) if defect_areas else 0, 2)
    
    # 3. DEEP FEATURES from trained model
    pred = trained_model.predict(np.expand_dims(img_normalized, axis=0), verbose=0)
    features['model_predictions'] = {
        'crazing': round(float(pred[0][0]), 4),
        'inclusion': round(float(pred[0][1]), 4),
        'patches': round(float(pred[0][2]), 4),
        'pitted_surface': round(float(pred[0][3]), 4),
        'rolled-in_scale': round(float(pred[0][4]), 4),
        'scratches': round(float(pred[0][5]), 4)
    }
    
    features['model_top_prediction'] = max(features['model_predictions'], 
                                           key=features['model_predictions'].get)
    
    return features

# =========================================
# 🧠 GROQ LLM CLASSIFICATION
# =========================================

def classify_with_groq_llm(image_features):
    """Use Groq LLM to classify based on extracted features"""
    
    prompt = f"""You are an expert steel surface defect classification system. Analyze the following extracted image features and determine the defect type.

IMAGE ANALYSIS FEATURES:
===========================

TEXTURE CHARACTERISTICS:
- Contrast Level: {image_features['contrast']} (0=uniform, high=very varied)
- Texture Entropy: {image_features['entropy']} (higher = more complex patterns)
- Edge Density: {image_features['edge_density']}% (percentage of edge pixels)

DEFECT REGION ANALYSIS:
- Number of Detected Regions: {image_features['num_defect_regions']}
- Average Defect Area: {image_features['avg_defect_area']} pixels²
- Total Defect Coverage: {image_features['total_defect_ratio']}%

NEURAL NETWORK PREDICTIONS:
- Crazing: {image_features['model_predictions']['crazing']:.4f}
- Inclusion: {image_features['model_predictions']['inclusion']:.4f}
- Patches: {image_features['model_predictions']['patches']:.4f}
- Pitted Surface: {image_features['model_predictions']['pitted_surface']:.4f}
- Rolled-in Scale: {image_features['model_predictions']['rolled-in_scale']:.4f}
- Scratches: {image_features['model_predictions']['scratches']:.4f}

DEFECT TYPE DESCRIPTIONS:
1. crazing - fine cracks, high entropy, many small regions
2. inclusion - particles, medium entropy, isolated regions
3. patches - discolored areas, medium contrast, bounded regions
4. pitted_surface - small holes, high edge density, many regions
5. rolled-in_scale - linear patterns, high contrast, medium coverage
6. scratches - linear marks, very high edge density, few regions

Return ONLY the defect type name (one word), nothing else."""
    
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=50
        )
        
        pred_label = response.choices[0].message.content.strip().lower()
        
        valid_classes = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
        for cls in valid_classes:
            if cls in pred_label:
                return cls
        
        return image_features['model_top_prediction']
        
    except Exception as e:
        return image_features['model_top_prediction']

# =========================================
# 🚀 MAIN PROCESSING PIPELINE
# =========================================

print("=" * 70)
print("🚀 GROQ LLM-BASED STEEL DEFECT CLASSIFICATION")
print("=" * 70)

image_files = get_all_images(IMAGES_DIR)[:100]

predictions_data = []
classification_results = {cls: {'count': 0, 'images': [], 'confidences': []} for cls in CLASS_NAMES}

print(f"\n📊 Processing {len(image_files)} images...\n")

for idx, img_info in enumerate(image_files, 1):
    filename = img_info['filename']
    img_path = img_info['full_path']
    true_label = img_info['defect_type'].lower() if img_info['defect_type'] else 'unknown'
    
    try:
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Extract features
        features = extract_image_features(img_rgb)
        
        # Groq LLM classification
        pred_label = classify_with_groq_llm(features)
        
        # Get confidence from model
        confidence = features['model_predictions'][pred_label]
        
        # Store results
        predictions_data.append({
            'filename': filename,
            'true_label': true_label,
            'pred_label': pred_label,
            'confidence': confidence,
            'features': features
        })
        
        classification_results[pred_label]['count'] += 1
        classification_results[pred_label]['images'].append((filename, img_rgb))
        classification_results[pred_label]['confidences'].append(confidence)
        
        match = "✅" if true_label == pred_label else "❌"
        print(f"[{idx:3d}] {filename:25s} | True: {true_label:18s} | LLM: {pred_label:18s} | Conf: {confidence:.4f} {match}")
        
    except Exception as e:
        print(f"[{idx:3d}] {filename:25s} | ❌ Error: {str(e)[:40]}")
        continue

# =========================================
# 📊 STATISTICS & VISUALIZATION
# =========================================

print(f"\n{'='*70}")
print(f"📊 CLASSIFICATION SUMMARY (Groq LLM-Based)")
print(f"{'='*70}\n")

total_predictions = len(predictions_data)
print(f"Total images analyzed: {total_predictions}\n")

print(f"{'Defect Type':<20} {'Count':>8} {'Percentage':>12} {'Avg Confidence':>16}")
print("-" * 58)

for cls in CLASS_NAMES:
    count = classification_results[cls]['count']
    percentage = (count / total_predictions * 100) if total_predictions > 0 else 0
    avg_conf = np.mean(classification_results[cls]['confidences']) if classification_results[cls]['confidences'] else 0
    print(f"{cls:<20} {count:>8} {percentage:>11.1f}% {avg_conf:>16.4f}")

# Create visualizations
print(f"\n{'='*70}")
print("🎨 Creating visualization...")
print(f"{'='*70}\n")

for defect_type in CLASS_NAMES:
    count = classification_results[defect_type]['count']
    
    if count == 0:
        continue
    
    sample_limit = min(9, count)
    fig = plt.figure(figsize=(15, 12))
    fig.suptitle(f"'{defect_type.upper()}' (Groq LLM) - {count} Images", 
                 fontsize=18, fontweight='bold', y=0.98)
    
    gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)
    
    for i in range(sample_limit):
        ax = fig.add_subplot(gs[i // 3, i % 3])
        img_display = classification_results[defect_type]['images'][i][0].astype(np.uint8)
        confidence = classification_results[defect_type]['confidences'][i]
        
        ax.imshow(img_display)
        ax.set_title(f"Conf: {confidence:.4f}", fontsize=11, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        
        for spine in ax.spines.values():
            spine.set_edgecolor('green')
            spine.set_linewidth(3)
    
    for i in range(sample_limit, 9):
        ax = fig.add_subplot(gs[i // 3, i % 3])
        ax.axis('off')
    
    plt.savefig(os.path.join(OUTPUT_DIR, f'groq_classified_{defect_type}.png'), dpi=150, bbox_inches='tight')
    print(f"✅ Saved: {defect_type.upper()} visualization")
    plt.close()

# Statistics chart
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Steel Defect Classification - GROQ LLM-Based", fontsize=16, fontweight='bold')

counts = [classification_results[cls]['count'] for cls in CLASS_NAMES]
colors_chart = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

bars1 = ax1.bar(CLASS_NAMES, counts, color=colors_chart, alpha=0.8, edgecolor='black', linewidth=2)
ax1.set_title('Number of Images per Defect Type', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count', fontsize=11)
ax1.tick_params(axis='x', rotation=45)
for bar, count in zip(bars1, counts):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}', 
            ha='center', va='bottom', fontweight='bold', fontsize=10)

percentages = [count / total_predictions * 100 for count in counts]
wedges, texts, autotexts = ax2.pie(percentages, labels=CLASS_NAMES, autopct='%1.1f%%',
                                     colors=colors_chart, startangle=90)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax2.set_title('Percentage Distribution', fontsize=12, fontweight='bold')

avg_confidences = [np.mean(classification_results[cls]['confidences']) if classification_results[cls]['confidences'] else 0 
                   for cls in CLASS_NAMES]
bars3 = ax3.bar(CLASS_NAMES, avg_confidences, color=colors_chart, alpha=0.8, edgecolor='black', linewidth=2)
ax3.set_title('Average Model Confidence', fontsize=12, fontweight='bold')
ax3.set_ylabel('Confidence', fontsize=11)
ax3.set_ylim(0, 1)
ax3.tick_params(axis='x', rotation=45)

ax4.boxplot([classification_results[cls]['confidences'] if classification_results[cls]['confidences'] else [0] 
             for cls in CLASS_NAMES], labels=CLASS_NAMES)
ax4.set_title('Confidence Distribution', fontsize=12, fontweight='bold')
ax4.set_ylabel('Confidence Score', fontsize=11)
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'groq_statistics.png'), dpi=150, bbox_inches='tight')
print(f"✅ Saved comprehensive statistics chart")
plt.show()

# Save results
results_json = {
    'method': 'Groq LLM-Based with Feature Extraction',
    'total_images': total_predictions,
    'classification_summary': {
        cls: {
            'count': classification_results[cls]['count'],
            'percentage': (classification_results[cls]['count'] / total_predictions * 100) if total_predictions > 0 else 0,
            'avg_confidence': float(np.mean(classification_results[cls]['confidences'])) if classification_results[cls]['confidences'] else 0
        }
        for cls in CLASS_NAMES
    }
}

results_path = os.path.join(OUTPUT_DIR, 'groq_llm_results.json')
with open(results_path, 'w') as f:
    json.dump(results_json, f, indent=2)

print(f"\n{'='*70}")
print(f"✅ All results saved to: {OUTPUT_DIR}")
print(f"{'='*70}\n")


ModuleNotFoundError: No module named 'skimage.stats'